# Load Data


In [5]:
import pandas as pd
import glob

files = glob.glob("../data/raw/202606-citibike-tripdata/*.csv")
df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

print(files)

/var/folders/dr/r55rrk2n0jsfmys_4px8ctxm0000gn/T/ipykernel_53826/3595554125.py:5: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
/var/folders/dr/r55rrk2n0jsfmys_4px8ctxm0000gn/T/ipykernel_53826/3595554125.py:5: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
/var/folders/dr/r55rrk2n0jsfmys_4px8ctxm0000gn/T/ipykernel_53826/3595554125.py:5: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
/var/folders/dr/r55rrk2n0jsfmys_4px8ctxm0000gn/T/ipykernel_53826/3595554125.py:5: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat((pd.read_csv(f) for f in files), ignore_index=

['../data/raw/202606-citibike-tripdata/202606-citibike-tripdata_6.csv', '../data/raw/202606-citibike-tripdata/202606-citibike-tripdata_5.csv', '../data/raw/202606-citibike-tripdata/202606-citibike-tripdata_4.csv', '../data/raw/202606-citibike-tripdata/202606-citibike-tripdata_1.csv', '../data/raw/202606-citibike-tripdata/202606-citibike-tripdata_3.csv', '../data/raw/202606-citibike-tripdata/202606-citibike-tripdata_2.csv']


In [ ]:
# Display basic information about the DataFrame
print(df.columns)
print(df.shape)

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='object')
(5384468, 13)


In [9]:
# Display basic information about the DataFrame
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5384468 entries, 0 to 5384467
Data columns (total 13 columns):
 #   Column              Dtype  
---  ------              -----  
 0   ride_id             object 
 1   rideable_type       object 
 2   started_at          object 
 3   ended_at            object 
 4   start_station_name  object 
 5   start_station_id    object 
 6   end_station_name    object 
 7   end_station_id      object 
 8   start_lat           float64
 9   start_lng           float64
 10  end_lat             float64
 11  end_lng             float64
 12  member_casual       object 
dtypes: float64(4), object(9)
memory usage: 534.0+ MB
None


In [10]:
# Check for missing values in the DataFrame
print(df.isna().sum())

ride_id                   0
rideable_type             0
started_at                0
ended_at                  0
start_station_name     3154
start_station_id       3154
end_station_name      13454
end_station_id        14211
start_lat              3154
start_lng              3154
end_lat               14190
end_lng               14190
member_casual             0
dtype: int64


In [ ]:
# Display value counts for 'member_casual' and 'rideable_type' columns, no unexpected categories found
print(df['member_casual'].value_counts())
print(df['rideable_type'].value_counts())

# Handle Missing Data

In [15]:
print(f"Before dropping: {df.shape[0]} rows")

Before dropping: 5384468 rows


In [16]:
# Confirm that missingness in start station name and coordinates is aligned (i.e., if one is missing, the other is also missing)
same_rows = (df['start_station_name'].isna() == df['start_lat'].isna()).all()
print(f"Start station name/coord missingness aligned: {same_rows}")

Start station name/coord missingness aligned: True


In [17]:
# Drop rows with missing start coordinates (can't map these trips)
df = df[df['start_lat'].notna() & df['start_lng'].notna()]
print(f"After dropping missing start coords: {df.shape[0]} rows")

# Drop rows with missing end coordinates (can't map these trips)
df = df[df['end_lat'].notna() & df['end_lng'].notna()]
print(f"After dropping missing end coords: {df.shape[0]} rows")

# Drop rows with missing member_casual (can't categorize these trips)
df = df[df['member_casual'].notna()]
print(f"After dropping missing member_casual: {df.shape[0]} rows")

After dropping missing start coords: 5381314 rows
After dropping missing end coords: 5367618 rows
After dropping missing member_casual: 5367618 rows


In [18]:
# Rows that have end coordinates but no end_station_id/name (dockless e-bike returns)
# Flag them instead of dropping, since we still have usable lat/lng
no_end_station = df['end_station_id'].isna() & df['end_lat'].notna()
df['end_is_dockless'] = no_end_station
print(f"Rows kept with coordinates but no end station (dockless returns): {no_end_station.sum()}")

print(f"\nFinal shape after Step 3: {df.shape}")

Rows kept with coordinates but no end station (dockless returns): 21

Final shape after Step 3: (5367618, 14)


# Parse Timestamps

# Compute Trip Duration

# Filter Our Obviously Bad Rows
eg. very short trips (under ~1 minute might be faulse starts or not real rides)

# Save Cleaned Output